# Allergen Model Evaluation (Thresholds on Validation, Held-Out Test)

**Purpose:** calibrate thresholds on the **validation** set, run the **single** held-out
**test** evaluation that becomes the final reported numbers, and compare the ML model
against the official OFF allergen tags as a real-world signal.

## Methodology

- The model was chosen in `08_allergen_training` (`models/model_selection.json`).
- Threshold strategies are compared on validation only; the selected samples-F1 policy is fixed before any held-out evaluation.
- The test set is evaluated **once** with those fixed thresholds — no iteration.
- The cross-validated samples-F1 threshold is the selected default; recall-first thresholds are saved as a safety-oriented alternative.
- **ML vs official tags** is the honest external check: how well does the model agree
  with manufacturer-declared allergens?


## Setup & Imports

In [1]:
import os
import json
import ast
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

from sklearn.metrics import f1_score, recall_score
from scipy.special import expit

from utils.data_utils import get_data_directories, load_labeled_data
from utils.text_processing import (
    get_allergen_list,
    clean_html,
    parse_official_tags,
    has_explicit_allergen_statement,
    rule_match,
)
from utils.model_utils import (
    load_model_and_tokenizer,
    find_best_thresholds,
    find_best_global_sample_threshold,
    cross_validate_sample_thresholds,
    apply_thresholds,
    predict_ml,
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis,
    evaluate_rule_vs_official,
)

print("Imports complete.")

Imports complete.


## 1. Load Chosen Model + Canonical Splits

In [2]:
dirs = get_data_directories()
split_dir = os.path.join(dirs["final"], "splits")

def load_split(name):
    split_path = os.path.join(split_dir, name)
    if not os.path.exists(split_path):
        raise FileNotFoundError(
            f"Missing split file {split_path}. Run Notebook 07 (training data preparation) first."
        )
    d = pd.read_csv(split_path)
    d["labels"] = d["labels"].apply(ast.literal_eval)
    if "text" not in d.columns:
        d["text"] = d["ingredients_cleaned"]
    return d

train_aug_df = load_split("train_augmented.csv")
val_df = load_split("val.csv")
def normalize_text(series):
    return series.fillna("").astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

train_text_keys = set(normalize_text(train_aug_df["text"]))
val_overlap = normalize_text(val_df["text"]).isin(train_text_keys)
print(f"Excluded {int(val_overlap.sum())} validation rows with exact augmented-train-text duplicates.")
val_df = val_df.loc[~val_overlap].reset_index(drop=True)
val_texts = val_df["text"].tolist()
val_labels = np.array(val_df["labels"].tolist())
allergen_list = get_allergen_list()

with open(os.path.join(dirs["models"], "model_selection.json")) as f:
    selection = json.load(f)
print(f"Chosen model: {selection['winner']}")

final_model_dir = os.path.join(dirs["models"], "mobilebert_allergen_final")
if not os.path.exists(final_model_dir):
    raise FileNotFoundError(
        f"Model directory not found: {final_model_dir}. Run Notebook 08 (allergen training) first."
    )
model, tokenizer, device = load_model_and_tokenizer(final_model_dir)
max_length = selection.get("max_len", 209)
print(f"Loaded canonical model (winner={selection['winner']}), max_length={max_length}")


Excluded 5 validation rows with exact augmented-train-text duplicates.
Chosen model: asl
Loaded canonical model (winner=asl), max_length=209


## 2. Prediction Helper (handles ensemble winner)

In [3]:
def predict_probs(texts, batch_size=8):
    if selection["winner"] == "ensemble":
        probs = []
        for seed in selection["models"]["ensemble_seeds"]:
            path = os.path.join(dirs["models"], "mobilebert_allergen_ensemble", f"seed_{seed}")
            m, tk, dev = load_model_and_tokenizer(path)
            _, p = predict_ml(texts, m, tk, dev, max_length=max_length, batch_size=batch_size)
            probs.append(p)
        return np.mean(probs, axis=0)
    _, probs = predict_ml(texts, model, tokenizer, device, max_length=max_length, batch_size=batch_size)
    return probs

## 3. Threshold Calibration on VALIDATION

In [4]:
val_probs = predict_probs(val_texts)

# Compare the sample-F1 objective against the recall-first allergen safety policy.
threshold_cv = cross_validate_sample_thresholds(
    val_probs,
    val_labels,
    n_splits=5,
    random_state=42,
)
best_t_global, best_global_samples_f1 = find_best_global_sample_threshold(val_probs, val_labels)
use_global_threshold = threshold_cv["tuned_mean"] > threshold_cv["fixed_mean"]
sample_f1_threshold = best_t_global if use_global_threshold else 0.5
sample_f1_thresholds = np.full(len(allergen_list), sample_f1_threshold, dtype=float)

recall_first_thresholds = find_best_thresholds(val_probs, val_labels, beta=2.0, min_recall=0.9)
balanced_thresholds = find_best_thresholds(val_probs, val_labels, beta=1.0, min_recall=None)
recall_first_preds = apply_thresholds(val_probs, recall_first_thresholds)
sample_f1_preds = apply_thresholds(val_probs, sample_f1_thresholds)

print("Five-fold validation samples F1:")
print(f"  Fixed 0.5: {threshold_cv['fixed_mean']:.4f} ± {threshold_cv['fixed_std']:.4f}")
print(f"  Fold-tuned global: {threshold_cv['tuned_mean']:.4f} ± {threshold_cv['tuned_std']:.4f}")
print(f"Selected samples-F1 policy: {'global' if use_global_threshold else 'fixed 0.5'}")
print(f"Full-validation global threshold: {best_t_global:.2f} (samples F1 {best_global_samples_f1:.4f})")
print("Per-class validation recall (samples-F1 vs recall-first):")
sample_recalls = recall_score(val_labels, sample_f1_preds, average=None, zero_division=0)
recall_first_recalls = recall_score(val_labels, recall_first_preds, average=None, zero_division=0)
for i, allergen in enumerate(allergen_list):
    print(f"  {allergen:<12} {sample_recalls[i]:.3f} vs {recall_first_recalls[i]:.3f}")
print(
    "Validation samples F1: "
    f"samples-focused={f1_score(val_labels, sample_f1_preds, average='samples', zero_division=0):.4f}, "
    f"recall-first={f1_score(val_labels, recall_first_preds, average='samples', zero_division=0):.4f}"
)

# Use the cross-validated sample-F1 policy; preserve recall-first thresholds as an alternative.
best_thresholds = sample_f1_thresholds
np.save(os.path.join(dirs["models"], "best_thresholds.npy"), best_thresholds)
np.save(os.path.join(dirs["models"], "best_thresholds_recall_first.npy"), recall_first_thresholds)
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "recall_first_thresholds": recall_first_thresholds.tolist(),
    "sample_f1_thresholds": sample_f1_thresholds.tolist(),
    "sample_f1_threshold": float(sample_f1_threshold),
    "sample_f1_threshold_policy": "global" if use_global_threshold else "fixed_0.5",
    "sample_f1_threshold_cv": {
        "fixed_mean": threshold_cv["fixed_mean"],
        "tuned_mean": threshold_cv["tuned_mean"],
        "fixed_std": threshold_cv["fixed_std"],
        "tuned_std": threshold_cv["tuned_std"],
    },
    "rule_conf_threshold": 0.5,
    "mode": "hard_override",
    "winner": selection["winner"],
    "max_length": max_length,
    "threshold_objective": "samples_f1_cv",
    "threshold_beta": 2.0,
    "threshold_min_recall": 0.9,
}
with open(os.path.join(dirs["models"], "hybrid_config.json"), "w") as f:
    json.dump(hybrid_config, f, indent=2)
print("Saved samples-F1 thresholds as default and recall-first thresholds as an alternative.")


Five-fold validation samples F1:
  Fixed 0.5: 0.6429 ± 0.0295
  Fold-tuned global: 0.6356 ± 0.0399
Selected samples-F1 policy: fixed 0.5
Full-validation global threshold: 0.50 (samples F1 0.6433)
Per-class validation recall (samples-F1 vs recall-first):
  milk         1.000 vs 1.000
  eggs         0.923 vs 0.923
  peanuts      0.909 vs 0.909
  tree_nuts    0.875 vs 1.000
  soy          0.970 vs 0.970
  wheat        0.936 vs 0.936
  fish         0.900 vs 0.900
  shellfish    1.000 vs 1.000
Validation samples F1: samples-focused=0.6433, recall-first=0.6211
Saved samples-F1 thresholds as default and recall-first thresholds as an alternative.


## 4. Held-Out Test Evaluation (single, final)

Thresholds were fixed on validation; the test set is evaluated once. These are the
numbers to report.

In [5]:
test_df = load_split("test.csv")
test_texts = test_df["text"].tolist()
test_labels = np.array(test_df["labels"].tolist())
test_probs = predict_probs(test_texts)
test_preds = apply_thresholds(test_probs, best_thresholds)

print("=" * 64)
print("  FINAL HELD-OUT TEST EVALUATION")
print("  (thresholds tuned on validation only; test used once)")
print("=" * 64)
print_classification_report(test_labels, test_preds, allergen_list, prefix="Test Set (optimized thresholds)")
print_per_class_metrics(test_labels, test_preds, allergen_list, prefix="Per-class on test:")
print(f"\nSamples F1: {f1_score(test_labels, test_preds, average='samples', zero_division=0):.4f}")

error_analysis(test_texts, test_labels, test_preds, allergen_list, max_errors=3)

  FINAL HELD-OUT TEST EVALUATION
  (thresholds tuned on validation only; test used once)
=== Test Set (optimized thresholds) ===
              precision    recall  f1-score   support

        milk     0.9333    0.9333    0.9333        75
        eggs     0.7368    0.9333    0.8235        15
     peanuts     0.9091    0.7143    0.8000        14
   tree_nuts     0.7000    0.8750    0.7778         8
         soy     0.7442    0.8889    0.8101        36
       wheat     0.9245    0.9423    0.9333        52
        fish     0.8333    0.7692    0.8000        13
   shellfish     1.0000    0.6250    0.7692         8

   micro avg     0.8640    0.8914    0.8775       221
   macro avg     0.8477    0.8352    0.8309       221
weighted avg     0.8737    0.8914    0.8779       221
 samples avg     0.5940    0.5921    0.5841       221

Macro F1: 0.8309
Micro F1: 0.8775


Per-class on test:
milk          Prec: 93.33%  Rec: 93.33%  F1: 93.33%
eggs          Prec: 73.68%  Rec: 93.33%  F1: 82.35%
peanuts

[7, 9, 17]

## 5. Non-Circular Ground-Truth Check: Hand-Labeled Hard Cases

Every F1 above compares predictions against labels that were **generated by the same rule
engine** we are evaluating — a form of circularity. This section breaks that loop:
`data/final/hard_cases.csv` holds **75 hard-case products** whose allergens were labeled
by a human (not the rule engine). They are drawn from the held-out test split and biased
toward the "difficult edge cases" the project cares about: facility cross-contamination
warnings, ambiguous Filipino terms, nut-free "nut" products (coconut, shea, water
chestnut, green tea), and chocolate/confectionery.

Labels follow the same **recall-first semantics** as the rest of the pipeline: a
"may contain X" cross-contamination warning counts as a positive (a risk is still a risk).

This is the non-circular reference: the only signal not derived from the rule engine being evaluated.

In [6]:
# Non-circular ground truth: 75 hand-labeled hard cases (human labels, not rule-engine generated).
# Human (not rule-engine) labels, recall-first semantics (may-contain = positive).
HC_PATH = os.path.join(dirs["final"], "hard_cases.csv")
hc = pd.read_csv(HC_PATH)
hc_texts = hc["text"].fillna("").tolist()
hc_labels = hc[allergen_list].to_numpy()  # (75, 8) — matches rule_preds_hc / ml_preds_hc

# Rule engine
rule_preds_hc = np.array([[1 if rule_match(t, a) else 0 for a in allergen_list] for t in hc_texts])
# ML with recall-first thresholds (fixed on validation, never on this set)
hc_probs = predict_probs(hc_texts, batch_size=8)
ml_preds_hc = apply_thresholds(hc_probs, best_thresholds)
# Hybrid = recall-first union (rule_priority for every allergen)
hybrid_preds_hc = np.maximum(rule_preds_hc, ml_preds_hc)


def prf(t, p):
    tp = int(((p == 1) & (t == 1)).sum())
    fp = int(((p == 1) & (t == 0)).sum())
    fn = int(((p == 0) & (t == 1)).sum())
    pr = tp / max(tp + fp, 1)
    re = tp / max(tp + fn, 1)
    return pr, re, (2 * pr * re / (pr + re) if (pr + re) > 0 else 0.0)


print(f"HARD-CASE EVAL vs HUMAN LABELS (N={len(hc)})")
print(f"{'Allergen':<10} {'Rule P/R/F1':>25} {'ML P/R/F1':>25} {'Hybrid P/R/F1':>25}")
for j, a in enumerate(allergen_list):
    row = []
    for preds in (rule_preds_hc, ml_preds_hc, hybrid_preds_hc):
        pr, re, f1 = prf(hc_labels[:, j], preds[:, j])
        row.append(f"{pr:.3f}/{re:.3f}/{f1:.3f}")
    print(f"{a:<10} {row[0]:>25} {row[1]:>25} {row[2]:>25}")

print("\nSamples F1 vs human labels "
      f"(rule/ML/hybrid): {f1_score(hc_labels, rule_preds_hc, average='samples', zero_division=0):.3f} / "
      f"{f1_score(hc_labels, ml_preds_hc, average='samples', zero_division=0):.3f} / "
      f"{f1_score(hc_labels, hybrid_preds_hc, average='samples', zero_division=0):.3f}")

j = allergen_list.index("tree_nuts")
print("\ntree_nuts FP/FN breakdown:")
for name, preds in [("Rule", rule_preds_hc), ("ML", ml_preds_hc), ("Hybrid", hybrid_preds_hc)]:
    pr, re, f1 = prf(hc_labels[:, j], preds[:, j])
    fp = int(((hc_labels[:, j] == 0) & (preds[:, j] == 1)).sum())
    fn = int(((hc_labels[:, j] == 1) & (preds[:, j] == 0)).sum())
    print(f"  tree_nuts {name:6s}: {fp} FP, {fn} FN  P={pr:.3f} R={re:.3f}")

HARD-CASE EVAL vs HUMAN LABELS (N=75)
Allergen                 Rule P/R/F1                 ML P/R/F1             Hybrid P/R/F1
milk               0.976/1.000/0.988         0.976/0.976/0.976         0.976/1.000/0.988
eggs               0.944/1.000/0.971         0.933/0.824/0.875         0.944/1.000/0.971
peanuts            1.000/1.000/1.000         1.000/0.909/0.952         1.000/1.000/1.000
tree_nuts          1.000/1.000/1.000         1.000/0.818/0.900         1.000/1.000/1.000
soy                1.000/0.978/0.989         1.000/0.652/0.789         1.000/0.978/0.989
wheat              0.981/0.981/0.981         0.981/0.962/0.971         0.981/0.981/0.981
fish               0.846/1.000/0.917         0.833/0.909/0.870         0.846/1.000/0.917
shellfish          1.000/1.000/1.000         1.000/0.875/0.933         1.000/1.000/1.000

Samples F1 vs human labels (rule/ML/hybrid): 0.914 / 0.856 / 0.914

tree_nuts FP/FN breakdown:
  tree_nuts Rule  : 0 FP, 0 FN  P=1.000 R=1.000
  tree_nuts ML   

## 6. ML vs Official OFF Tags (real-world signal)

In [7]:
# External check runs on the held-out test split only.
# Earlier versions scored the ML model against OFF tags on ALL 1057 products, which
# includes the 737 products the model was trained on (in-sample → inflated agreement).
# Restricting to the 158 held-out test products provides a non-circular signal:
# neither the rule engine nor the training labels are the ground truth here — the
# manufacturer-declared `allergens_tags` from Open Food Facts are.
CLEANED_PATH = os.path.join(dirs["processed"], "cleaned_dataset.csv")
df_off = test_df.copy()  # already holds code + text
off_tags = load_labeled_data(CLEANED_PATH)[["code", "allergens_tags"]]
df_off = df_off.merge(off_tags, on="code", how="left")

ml_probs_test = predict_probs(df_off["text"].tolist(), batch_size=8)
ml_preds_test = apply_thresholds(ml_probs_test, best_thresholds)
df_off["detected_allergens"] = [
    [allergen_list[i] for i, v in enumerate(row) if v == 1] for row in ml_preds_test
]
df_off["official_allergens_mapped"] = df_off["allergens_tags"].apply(
    lambda x: parse_official_tags(x) if pd.notna(x) and str(x) not in ("[]", "nan", "") else []
)

full_results = evaluate_rule_vs_official(df_off, allergen_list, prefix="ML vs OFF tags (held-out test only)")

df_off["explicit"] = df_off["ingredients_text_en"].apply(has_explicit_allergen_statement)
explicit_results = evaluate_rule_vs_official(
    df_off[df_off["explicit"]], allergen_list, prefix="ML vs OFF tags (explicit statements, held-out)"
)
print(f"\nProducts with explicit statements: {int(df_off['explicit'].sum())}")


  ML vs OFF tags (held-out test only)
  Samples evaluated: 158
  Exact match (strict):        65.19%
  Detected ⊆ official (no FP): 84.18%
  Detected ⊇ official (no FN): 79.75%
  Avg Jaccard:                 80.82%

  Per-class metrics:
  Allergen      Precision     Recall         F1
  ────────────────────────────────────────────
  milk            88.00%    88.00%    88.00%
  eggs            73.68%    87.50%    80.00%
  peanuts         54.55%    60.00%    57.14%
  tree_nuts       40.00%    57.14%    47.06%
  soy             86.05%    68.52%    76.29%
  wheat           86.79%    83.64%    85.19%
  fish            66.67%    80.00%    72.73%
  shellfish       80.00%    66.67%    72.73%


  ML vs OFF tags (explicit statements, held-out)
  Samples evaluated: 11
  Exact match (strict):        63.64%
  Detected ⊆ official (no FP): 81.82%
  Detected ⊇ official (no FN): 72.73%
  Avg Jaccard:                 83.44%

  Per-class metrics:
  Allergen      Precision     Recall         F1
  ────────

## 7. Seed Variance (if the ensemble won)

In [8]:
if selection["winner"] == "ensemble":
    print("Per-seed test samples F1 (ensemble averaging should beat most members):")
    for seed in selection["models"]["ensemble_seeds"]:
        path = os.path.join(dirs["models"], "mobilebert_allergen_ensemble", f"seed_{seed}")
        m, tk, dev = load_model_and_tokenizer(path)
        _, p = predict_ml(test_texts, m, tk, dev, max_length=max_length, batch_size=8)
        preds = apply_thresholds(p, best_thresholds)
        print(f"  Seed {seed:4d}: samples F1 {f1_score(test_labels, preds, average='samples', zero_division=0):.4f}")
    print(f"  Ensemble : samples F1 {f1_score(test_labels, test_preds, average='samples', zero_division=0):.4f}")
else:
    print(f"Single-model winner ({selection['winner']}) — per-seed spread not applicable.")

Single-model winner (asl) — per-seed spread not applicable.


## Calibration note

`utils.model_utils.TemperatureScaling` is available if you want post-hoc probability
calibration: learn a temperature on the validation logits and re-scale before thresholding.
It was left out of the main flow to keep the reported numbers on a single, well-defined
thresholding pipeline.

**Next:** run `10_hybrid_evaluation.ipynb`.